# GRPO su DeepMath-103K con TRL

Questo notebook spiega gli oggetti coinvolti nel training GRPO usando il dataset ufficiale di TRL.
A differenza di GSM8K, **DeepMath-103K è già nel formato che `GRPOTrainer` si aspetta** — nessun preprocessing.

In [9]:
# !pip install -q trl datasets math_verify

## 1. Il dataset — struttura

In [1]:
from datasets import load_dataset

dataset = load_dataset("trl-lib/DeepMath-103K", split="train")
print(dataset)
# Dataset con solo 2 colonne: 'prompt' e 'solution'

/work/fis1/RT-DeepRL/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['prompt', 'solution'],
    num_rows: 97870
})


In [2]:
# Colonne già nel formato GRPOTrainer: nessuna trasformazione necessaria
print("Colonne:", dataset.column_names)
print()
example = dataset[0]

# 'prompt' è una lista di messaggi chat (formato conversazionale HuggingFace)
print("PROMPT:", example["prompt"])
print()
# 'solution' è la risposta attesa come stringa grezza — passata as-is alla reward function
print("SOLUTION:", example["solution"])

Colonne: ['prompt', 'solution']

PROMPT: [{'content': 'Is it possible to construct an uncountable set of subsets of the positive integers, \\( T \\), such that for any two subsets \\( C, D \\) in \\( T \\), either \\( C \\) is a subset of \\( D \\) or \\( D \\) is a subset of \\( C \\)? Provide a justification for your answer.', 'role': 'user'}]

SOLUTION: Yes


In [3]:
# Guardiamo qualche solution per capire il formato
for i in range(10):
    print(f"[{i}] solution={dataset[i]['solution']!r}")

[0] solution='Yes'
[1] solution='$1$'
[2] solution='$0$'
[3] solution='$\\frac{1}{5}$'
[4] solution='Yes'
[5] solution='$2^{p_n}$'
[6] solution='No'
[7] solution='$0$'
[8] solution='$\\dfrac{\\beta}{a}$'
[9] solution='Yes'


## 2. `accuracy_reward` — firma e comportamento

```python
accuracy_reward(completions, solution, **kwargs) -> list[float | None]
```

| Parametro | Chi lo fornisce | Tipo |
|-----------|----------------|------|
| `completions` | il trainer (generazioni del modello) | `list[list[dict]]` |
| `solution` | il dataset (colonna `solution`) | `list[str]` |

Il trainer passa automaticamente come `**kwargs` tutte le colonne del dataset che non sono `prompt`.
Siccome il dataset ha solo `prompt` e `solution`, e `accuracy_reward` accetta `solution` come argomento posizionale, **il collegamento è automatico per nome colonna**.

Internamente usa `math_verify` per parsing LaTeX/numerico e confronto simbolico.

In [8]:
from trl.rewards import accuracy_reward

# Simuliamo una chiamata manuale
fake_completions = [
    [{"role": "assistant", "content": r"The answer is \boxed{Yes}"}],
    [{"role": "assistant", "content": r"The answer is \boxed{No}"}],
    [{"role": "assistant", "content": r"\boxed{\frac{1}{3}}"}],
    [{"role": "assistant", "content": r"\boxed{\frac{1}{2}}"}],
]
fake_solutions = ["Yes", "Yes", r"\frac{1}{3}", r"\frac{1}{3}"]

rewards = accuracy_reward(fake_completions, fake_solutions)
for comp, sol, r in zip(fake_completions, fake_solutions, rewards):
    content = comp[0]["content"]
    print(f"completion: {content!r:<45}  solution: {sol!r:<15}  reward: {r}")

completion: 'The answer is \\boxed{Yes}'                   solution: 'Yes'            reward: None
completion: 'The answer is \\boxed{No}'                    solution: 'Yes'            reward: None
completion: '\\boxed{\\frac{1}{3}}'                        solution: '\\frac{1}{3}'   reward: 1.0
completion: '\\boxed{\\frac{1}{2}}'                        solution: '\\frac{1}{3}'   reward: 0.0


**Note:**
- Reward `1.0` = corretto, `0.0` = sbagliato, `None` = gold non parseable (esempio skippato dal trainer)
- `accuracy_reward` cerca `\boxed{...}` nella completion — il prompt dovrebbe chiedere al modello di usare questo formato

## 3. `GRPOTrainer` — il loop

GRPO per ogni batch:
1. Genera **G completions** per ogni prompt (`num_generations`)
2. Calcola la reward con `reward_funcs` per ognuna
3. Normalizza le reward nel gruppo → advantage
4. Policy gradient con clip KL dalla reference policy

In [ ]:
import torch
from trl import GRPOTrainer, GRPOConfig

training_args = GRPOConfig(
    output_dir="grpo-deepmath",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,        # G: completions per prompt per calcolare il vantaggio
    max_prompt_length=512,
    max_completion_length=1024,
    learning_rate=1e-6,
    logging_steps=10,
    bf16=torch.cuda.is_available(),
)

trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=accuracy_reward,
    args=training_args,
    train_dataset=dataset,
)

print("Colonne dataset:", trainer.train_dataset.column_names)
print("Reward functions:", [f.__name__ for f in trainer.reward_funcs])

## 4. Training

In [ ]:
trainer.train()

## Snippet finale — identico all'esempio TRL ufficiale

In [ ]:
from datasets import load_dataset
from trl import GRPOTrainer
from trl.rewards import accuracy_reward

dataset = load_dataset("trl-lib/DeepMath-103K", split="train")

trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=accuracy_reward,
    train_dataset=dataset,
)
trainer.train()